# VLM Modality Benchmark — Kaggle Edition

Kaggle gives **30 hours/week** of free GPU (T4 or P100, 16GB VRAM).
Session limit: 12 hours. Run 2-3 models per session.

**Setup:**
1. Enable GPU: Settings → Accelerator → GPU T4 x2 (or P100)
2. Enable Internet: Settings → Internet → On
3. Change `MODELS_TO_RUN` each session
4. Results save to `/kaggle/working/` (auto-saved as output)

In [ ]:
!pip install torch transformers datasets scipy pyyaml tqdm pillow bitsandbytes torchvision --quiet

In [ ]:
!git clone https://github.com/Ro-netizen004/vlm-modality-research.git /kaggle/working/repo 2>/dev/null || echo 'exists'
import sys
sys.path.insert(0, '/kaggle/working/repo')

OUTPUT_DIR = '/kaggle/working/results'

In [ ]:
# ══════════════════════════════════════════════════════════
#  CHANGE THIS EACH SESSION
# ══════════════════════════════════════════════════════════
MODELS_TO_RUN = [0, 1, 2]  # indices into ALL_MODELS
NUM_PROBLEMS = None  # None = full 1319

ALL_MODELS = [
    {'name': 'Qwen/Qwen2-VL-2B-Instruct', 'type': 'qwen',
     'max_new_tokens': 256, 'torch_dtype': 'bfloat16', 'quantize': None},
    {'name': 'llava-hf/llava-v1.6-mistral-7b-hf', 'type': 'llava',
     'max_new_tokens': 512, 'torch_dtype': 'float16', 'quantize': '4bit'},
    {'name': 'Qwen/Qwen2.5-VL-7B-Instruct', 'type': 'qwen',
     'max_new_tokens': 512, 'torch_dtype': 'bfloat16', 'quantize': '4bit'},
    {'name': 'OpenGVLab/InternVL2-8B', 'type': 'internvl',
     'max_new_tokens': 512, 'torch_dtype': 'bfloat16', 'quantize': '4bit'},
    {'name': 'llava-hf/llava-onevision-qwen2-7b-ov-hf', 'type': 'llava_onevision',
     'max_new_tokens': 512, 'torch_dtype': 'float16', 'quantize': '4bit'},
    {'name': 'microsoft/Phi-3.5-vision-instruct', 'type': 'phi',
     'max_new_tokens': 512, 'torch_dtype': 'bfloat16', 'quantize': '4bit'},
    {'name': 'openbmb/MiniCPM-V-2_6', 'type': 'minicpm',
     'max_new_tokens': 512, 'torch_dtype': 'bfloat16', 'quantize': '4bit'},
    {'name': 'HuggingFaceM4/Idefics3-8B-Llama3', 'type': 'idefics',
     'max_new_tokens': 512, 'torch_dtype': 'bfloat16', 'quantize': '4bit'},
]

SELECTED = [ALL_MODELS[i] for i in MODELS_TO_RUN]
print(f'This session: {[m["name"].split("/")[-1] for m in SELECTED]}')

In [ ]:
import os, time, json, torch
import pandas as pd
from collections import Counter
from datasets import load_dataset
from tqdm import tqdm

from src.models import VLMModel
from src.evaluation import *
from src.rendering import render_all_images, load_image
from src.visualization import plot_error_breakdown, plot_mismatch_dominance

torch.manual_seed(42)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load dataset
ds = load_dataset('gsm8k', 'main', split='test')
if NUM_PROBLEMS:
    ds = ds.select(range(NUM_PROBLEMS))
questions = list(ds['question'])
references = list(ds['answer'])
N = len(questions)
print(f'Problems: {N}')

IMAGE_DIR = os.path.join(OUTPUT_DIR, 'rendered_images')
render_all_images(questions, IMAGE_DIR)

# ── Run each model ──
for mc in SELECTED:
    short = mc['name'].split('/')[-1]
    model_dir = os.path.join(OUTPUT_DIR, short)
    if os.path.exists(os.path.join(model_dir, 'statistics.json')):
        print(f'SKIP {short} (already done)'); continue
    os.makedirs(model_dir, exist_ok=True)
    print(f"\n{'='*70}\n  {mc['name']}\n{'='*70}")
    
    vlm = VLMModel(**{k:v for k,v in mc.items()})
    vlm.load()
    t0 = time.time()
    
    # Cond 1
    tp, tc, te = [], [], []
    for q, ref in tqdm(zip(questions, references), total=N, desc='Text'):
        try: p = vlm.generate_text_only(q)
        except Exception as e: p = f'ERROR: {e}'
        tp.append(p); tc.append(answers_match(p, ref)); te.append(classify_error(p, ref))
    print(f'  Text: {compute_accuracy(tc):.3f}')
    
    # Cond 2
    ip, ic, ie = [], [], []
    for i in tqdm(range(N), desc='Image'):
        try: p = vlm.generate_with_image(load_image(i, IMAGE_DIR))
        except Exception as e: p = f'ERROR: {e}'
        ip.append(p); ic.append(answers_match(p, references[i])); ie.append(classify_error(p, references[i]))
    print(f'  Image: {compute_accuracy(ic):.3f}')
    
    # Cond 3
    mp, mf, mid, mtd = [], [], [], []
    for i in tqdm(range(N), desc='Mismatch'):
        ti = (i+1)%N
        prompt = f"Solve the following math problem step by step. End with '#### <answer>'.\n\nProblem: {questions[ti]}"
        try: p = vlm.generate_with_image(load_image(i, IMAGE_DIR), text_prompt=prompt)
        except Exception as e: p = f'ERROR: {e}'
        pv, iv, tv = extract_numeric_answer(p), extract_numeric_answer(references[i]), extract_numeric_answer(references[ti])
        if pv is None or iv is None or tv is None:
            f, di, dt = 'invalid', None, None
        else:
            di, dt = abs(pv-iv), abs(pv-tv)
            f = 'image' if di<dt else ('text' if dt<di else 'equal')
        mp.append(p); mf.append(f); mid.append(di); mtd.append(dt)
    
    elapsed = time.time()-t0
    stats = compute_all_statistics(tc, ic, mf)
    stats['elapsed_minutes'] = round(elapsed/60, 1)
    print(format_statistics_report(stats))
    
    # Save
    pd.DataFrame({'problem_id':range(N),'question':questions,'reference':references,
        'pred_text':tp,'correct_text':tc,'error_text':te,
        'pred_rendered':ip,'correct_rendered':ic,'error_rendered':ie,
        'pred_mismatch':mp,'mismatch_follows':mf,
    }).to_csv(os.path.join(model_dir,'gsm8k_results.csv'), index=False)
    with open(os.path.join(model_dir,'statistics.json'),'w') as f:
        json.dump({k:(list(v) if isinstance(v,tuple) else v) for k,v in stats.items()}, f, indent=2)
    plot_error_breakdown({'Text-Only':te,'Rendered Image':ie}, mc['name'], model_dir)
    plot_mismatch_dominance(Counter(mf), mc['name'], model_dir)
    print(f'Saved: {model_dir}')
    vlm.unload()

print('\nDone! Results in /kaggle/working/results/ — auto-saved as notebook output.')